## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../systems/openmm/17-ala.openmm.zip")
simulation.load()

OmniRunner.close_all_runners()
imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: moveable restraints")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

## Commands

In [3]:
import numpy as np
from nanover.utilities.transforms import Transform

def get_first_avatar_transform():
    avatar = next((value for key, value in utilities.copy_shared_state().items() if key.startswith("avatar.")))
    head = next((value for value in avatar["components"] if "headset" in value["name"]))
    transform = Transform.from_state_transform([*head["position"], *head["rotation"]])
    return transform

def box_identity():
    utilities.scene_transform = Transform.identity().flipped

def box_origin():
    center = utilities.box_size
    utilities.scene_transform = Transform.from_trs(center * -.5).flipped

def box_head():
    center = utilities.box_size
    avatar = get_first_avatar_transform().to_state_transform()
    avatar_pos, avatar_rot = avatar[0:3], avatar[3:7]

    scaling = Transform.from_trs(scale=[.25, .25, .25])
    centering = Transform.from_trs(translation=center * -.5)
    flipping = Transform.x_axis_flip()
    avataring = Transform.from_trs(translation=avatar_pos, rotation=avatar_rot)

    final = Transform.from_stack(avataring, flipping, scaling, centering)

    utilities.scene_transform = final


utilities.define_command("user/box/identity", handler=box_identity, label="identity", icon="🧭")
utilities.define_command("user/box/origin", handler=box_origin, label="around", icon="⏺️")
utilities.define_command("user/box/head", handler=box_head, label="head", icon="😎")

In [4]:
utilities.show_logging()

Output()

In [5]:
utilities.scene_transform

<Transform of rotate[-180.0deg, 0.0deg, -0.0deg] scale[-1.0, -1.0, -1.0]>